---

## **DIPLOME UNIVERSITAIRE**
## **SORBONNE DATA ANALYTICS**
---
## **Projet Machine Learning**
## **Classification des genres musicaux**

---
*Professeur* :\
FAIK Lina\
linafaik08@gmail.com\
\
*Etudiants* :\
KOENIG Camille\
camille.koenig@gmail.com\
MERCHAN Diego\
diegomerchanm@gmail.com\
PHAN NGUYEN Jayson\
jaysonphannguyenpro@gmail.com\
BIZOT Xia\
xiabizot@free.fr


Promotion 007

Mars 2026

---
**Jeu de données kaggle : FMA : A Dataset For Music Analysis – Format small**

https://www.kaggle.com/datasets/imsparsh/fma-free-music-archive-small-medium

---


---

# Extraction features V2 — FMA Small
## Pipeline audio → tabulaire

---

## Objectif

Ce notebook produit le **dataset final `features_V2.csv`** utilisé pour toute la modélisation (Phase 3).  
Il est reproductible, idempotent (checkpoint : relançable sans réextraire), et traçable (logs erreurs).


---

## Décisions héritées du notebook EDA (figées)

- **Périmètre** : FMA Small — 8 000 MP3
- **Standardisation audio** : mono, sr=22 050 Hz, fenêtre=30s
- **Features cibles** : 10 groupes × 7 stats + tempo → 351 features
- **Labels** : genre_top (tâche A) + genres_decoded + mismatch (tâche B + analyse transverse)
- **Split** : group split par artiste (réalisé dans le notebook modélisation)
- **Métrique** : macro F1 (rappel — pas utilisée ici)

---

## Stratégie d'extraction

1. Construire `labels_df` complet depuis `tracks.csv` + `genres.csv` (tous les labels nécessaires à la Phase 3)
2. Extraire features maison sur 500 pistes (échantillon seed=42) → valider pipeline par corrélation avec FMA
3. Documenter l'écart MFCC (normalisation corpus-level FMA vs piste-level maison)
4. Extraire les 8 000 pistes complètes avec checkpoint toutes les 200 pistes
5. Joindre labels + features → exporter `features_V2.csv`
---

## Structure du notebook

| Cellule | Section | Contenu |
|---------|---------|---------|
| C5–C7 | 1. Configuration | Imports (C5), chemins (C6), versions + seed + stratégies (C7) |
| C8 | 2. Labels | tracks.csv + genres.csv → labels_df 11 colonnes + subgenres_eligible |
| C9 | 3. Validation environnement | librosa, décodage audio |
| C10 | 4. Fonctions extraction | extract_features, run_extraction |
| C11 | 5. Test unitaire | Validation schéma 1 piste |
| C12 | 6. Test pipeline 500 pistes | Extraction maison échantillon |
| C13 | 7. Corrélation FMA | Validation pipeline vs FMA |
| C14 | 8. Extraction complète | 8 000 pistes + checkpoint |
| C15 | 9. Retry ciblé | Pistes manquantes / erreurs |
| C16 | 10. Construction CSV final | Jointure labels + features |
| C17 | 11. Export + validation | Sauvegarde features_V2.csv |

---

# Extraction features V2 : des MP3 bruts au dataset tabulaire

## 0. Objectif du notebook

Ce notebook est un **livrable de pipeline (Phase 2)**. Il couvre la chaîne complète depuis les MP3 bruts jusqu'au **dataset final `features_V2.csv`** prêt pour la modélisation (Phase 3).

Il est conçu pour être :
- **reproductible** (chemins relatifs, seed=42, versions loggées),
- **idempotent** (relançable sans réextraire : le resume détecte les pistes déjà extraites depuis le CSV écrit),
- **traçable** (quality gates go/no-go explicites, logs erreurs, diagnostic corrélation FMA),
- **orienté livrable** : produire `features_V2.csv` = 351 features audio + tous les labels Phase 3.

Le fil directeur est identique à l'EDA : justifier chaque décision technique et transmettre un dataset exploitable sans ambiguïté à l'équipe modélisation.

---

## 1. Configuration

### 1.1. Imports
- audio : `librosa`, `librosa.display`
- données : `pandas`, `numpy`, `scipy.stats`
- système : `pathlib`, `os`, `time`, `ast`, `traceback`
- visualisation : `matplotlib` (validation uniquement)

### 1.2. Chemins relatifs & organisation
Principe identique à l'EDA : tout est relatif au dossier du notebook. Les données brutes ne sont jamais modifiées.

- `BASE = Path.cwd()` → dossier PROJET/
- `AUDIO_DIR` → `data/raw/fma_small/fma_small/` (8 000 MP3)
- `META_DIR` → `data/raw/fma_metadata/` (tracks.csv, genres.csv, features.csv)
- `OUTPUT_DIR` → `outputs/features/` (créé automatiquement si absent)

Fichiers produits :
- `features_house_test500.csv` — extraction échantillon validation
- `features_house_full.csv` — extraction complète 8 000 pistes
- `features_house_full_errors.csv` — log MP3 en erreur
- `features_V2.csv` — **livrable final** (features + labels)

### 1.3. Paramètres figés (hérités EDA C27 — ne pas modifier)
- `SR = 22050`, `MONO = True`, `DURATION = 30.0`, `SEED = 42`
- `N_MFCC = 20`, `BATCH_SIZE = 200`, `N_TEST = 500`
- `SUBGENRES_STRATEGY` : top-k (k=3), multi-label si temps, seuil freq=50
- `SPLIT_STRATEGY` : group split par artiste, test_size=0.2, seed=42

---

## 2. Labels (tracks.csv + genres.csv)

Construction de `labels_df` complet **avant toute extraction** — c'est la source de vérité pour tous les labels Phase 3. Construire les labels après coup oblige à une jointure post-extraction qui peut introduire des désalignements.

| Colonne | Source | Usage Phase 3 |
|---------|--------|---------------|
| `genre_top` | tracks.csv | Variable cible tâche A (8 classes) |
| `genres` | tracks.csv | IDs bruts — traçabilité / audit |
| `genres_decoded` | genres.csv décodage | Variable cible tâche B (top-k / multi-label) |
| `n_subgenres` | calculé | Filtrage seuil freq_threshold=50 |
| `mismatch` | calculé | Analyse transverse — indicateur d'ambiguïté |
| `artist_name` | tracks.csv | Group split + analyse biais artiste |
| `track_title` | tracks.csv | Exemples qualitatifs rapport |
| `year` | tracks.csv date_created | Documentation biais temporel |
| `duration` | tracks.csv | Validation fenêtre 30s |
| `bit_rate` | tracks.csv | Détection outliers audio |

`genres_decoded` est stocké en string dans le CSV (`str(list)`). Phase 3 : utiliser `ast.literal_eval` pour récupérer la liste.

---

## 3. Validation environnement audio

Gate bloquante identique à l'EDA (C12) — `audio_decode_ok` doit être `True` avant toute extraction.  
Construction du `mp3_map` via `os.walk` (plus robuste que `rglob` sur Windows).  
Échantillon déterministe (seed=42, tri préalable) sur 5 pistes.

---

## 4. Fonctions extraction

### `compute_stats(arr)`
7 stats identiques à FMA : `mean`, `std`, `min`, `max`, `median`, `skew`, `kurtosis`.  
Résument la distribution complète du signal sur la fenêtre 30s — `skew` et `kurtosis` capturent asymétrie et aplatissement, discriminants pour les genres percussifs.

### `extract_features(path)`
10 extracteurs couvrant les 7 dimensions acoustiques → **351 features** par piste :

- **Timbre** : MFCC (caractère “couleur” du son)  
- **Harmonie** : tonnetz
- **Tonalité** : chroma
- **Texture / spectre** : centroid, bandwidth, rolloff, contrast  
- **Énergie** : RMSE  
- **Percussif** : ZCR  
- **Rythme** : tempo  

> `tonnetz` : calculé sur signal harmonique (`librosa.effects.harmonic`) — NaN si signal insuffisant, géré par try/except (pas de crash pipeline).  
> `tempo` : via `beat_track` (compatible librosa 0.11) — scalaire, pas de stats temporelles.

### `run_extraction(ids, ...)`
Runner générique : resume via CSV écrit (idempotent), batch append toutes les `BATCH_SIZE` pistes, log erreurs avec traceback, affichage progression et ETA.

---

## 5. Test unitaire (1 piste)

Validation du schéma sur 1 piste avant tout lancement :
- nombre de features == `EXPECTED_N_FEATURES` (351)
- tous les types == `float`
- NaN uniquement autorisés sur `tonnetz`

Gate bloquante — stoppe le pipeline si le schéma est incorrect.

---

## 6. Test pipeline (500 pistes)

Extraction sur 500 pistes déterministes (seed=42, tri préalable).  
Valide le pipeline en conditions réelles avant de lancer les 8 000 pistes.  
Idempotent : si `features_house_test500.csv` existe et est complet, le resume ne recalcule rien.

---

## 7. Corrélation FMA

Comparaison maison vs FMA sur les 500 pistes test — validation que nos features sont cohérentes avec la référence.

Résultats observés : corrélations > 0.85 sur la majorité des groupes (rmse, centroid, rolloff, zcr).  
Écart MFCC (~0.49) documenté et expliqué : FMA a calculé ses features sur le corpus complet (106 574 pistes) avec une normalisation corpus-level non reproductible. Non bloquant : `StandardScaler` en Phase 3 neutralise les décalages absolus.

---

## 8. Extraction complète (8 000 pistes)

Extraction complète avec checkpoint toutes les 200 pistes — perte max en cas de crash : 200 pistes.  
Resume automatique depuis `features_house_full.csv` si interruption.  
Validation post-extraction : diagnostic go/no-go, pistes manquantes, erreurs loggées.

---

## 9. Retry ciblé

Retry automatique sur les pistes manquantes et en erreur.  
Les MP3 corrompus résiduels sont documentés dans `features_house_full_errors.csv` — non bloquants pour la modélisation.

---

## 10. Construction du CSV final

Jointure `features_house_full.csv` + `labels_df` sur `track_id`.  
Ordre des colonnes : `track_id` | labels (11 colonnes) | features audio (351 colonnes).  
Validation : shape, doublons, NA par groupe, distribution `genre_top`, mismatch global.

---

## 11. Export + validation finale

Export `features_V2.csv` + rapport de validation complet.  
Vérification reproductibilité : rechargement depuis disque et comparaison shape/colonnes.

---

---

## Justification technique — Features V2 (351) et choix de représentation

---

Ce bloc justifie le design V2 des features (V1 = features formatées dans le datase) : **familles de features**, **résumé statistique tabulaire**, **choix de 7 statistiques**.
L’objectif est un vecteur fixe par piste (ML tabulaire), cohérent avec une comparaison DL séparée (CNN sur log-mel).

### 1) Familles de descripteurs (couverture multi-axe)
Les genres se distinguent sur plusieurs dimensions du signal. Features V2 combine :
- **Timbre** : MFCC (20) — couleur/texture globale (instrumentation).
- **Harmonie / tonalité** : chroma (12) + tonnetz (6) — structure harmonique.
- **Spectre / texture** : centroid, bandwidth, rolloff, contrast — brillance/étendue/rugosité.
- **Énergie** : RMSE — dynamique.
- **Percussif** : ZCR — transients/attaques.
- **Rythme** : tempo — indice rythmique explicite.

But : éviter une représentation “mono-feature” (MFCC seuls) peu robuste sur des genres poreux/hybrides.

### 2) Pourquoi un résumé statistique (tabulaire) plutôt que la séquence ?
Les descripteurs sont calculés **frame par frame** → on obtient des séries temporelles de valeurs.  
On les agrège en statistiques pour obtenir un **vecteur fixe/piste**, exportable (CSV), stable et reproductible (LR/SVM/arbres).  
Le DL (log-mel + CNN) ne dépend pas de cette agrégation : il apprend sur la représentation temps–fréquence.

### 3) Pourquoi 7 statistiques (pas seulement mean/std)
En audio, les distributions sont souvent non gaussiennes (transients, ruptures).  
`mean/std` ne suffisent pas à décrire extrêmes et forme de distribution.  
On utilise donc : `mean`, `std`, `min`, `max`, `median`, `skew`, `kurtosis`.  
Le **tempo** est scalaire → pas d’agrégation → 1 valeur.

### 4) Comptage : EXPECTED_N_FEATURES = 351
| Groupe | Famille | Dims | × Stats | Total |
|--------|---------|------|---------|-------|
| `mfcc` | Timbre | 20 | ×7 | 140 |
| `chroma_stft` | Harmonie | 12 | ×7 | 84 |
| `spectral_centroid` | Spectre | 1 | ×7 | 7 |
| `spectral_bandwidth` | Spectre | 1 | ×7 | 7 |
| `spectral_rolloff` | Spectre | 1 | ×7 | 7 |
| `spectral_contrast` | Texture | 7 | ×7 | 49 |
| `rmse` | Énergie | 1 | ×7 | 7 |
| `tonnetz` | Tonalité | 6 | ×7 | 42 |
| `zcr` | Percussif | 1 | ×7 | 7 |
| `tempo` | Rythme | 1 | ×1 | 1 |
| **Total** | | **51** | | **351** |

Total = **351** (hors `track_id` et labels).

### 5) Paramètres figés (contrat Phase 2 → Phase 3)
`sr=22050`, `mono=True`, `window_s=30`, `seed=42` sont figés pour garantir comparabilité, coût maîtrisé et reproductibilité.

Notes:
- `tonnetz` (harmonic) peut produire des NaN si signal insuffisant : géré par try/except dans `extract_features`.
- Le cadrage suggère 80–120 colonnes (mean/std). Ici, on assume 351 (7 stats) pour mieux capturer la distribution complète et tester l’impact sur la performance.

In [ ]:
# C5
# 1.1. Imports

from pathlib import Path
import os
import time
import json
import ast
import traceback
from typing import Dict, List, Tuple, Optional

import pandas as pd
import numpy as np
from scipy import stats as scipy_stats

import matplotlib.pyplot as plt

import librosa
import librosa.display

In [ ]:
# C6
# 1.2. Chemins relatifs (pipeline) — ROBUSTE & COURT

from pathlib import Path

BASE       = Path.cwd()
RAW_DIR    = BASE / "data" / "raw"
AUDIO_DIR  = RAW_DIR / "fma_small" / "fma_small"
META_DIR   = RAW_DIR / "fma_metadata"
OUTPUT_DIR = BASE / "outputs" / "features"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRACKS_CSV  = META_DIR / "tracks.csv"
GENRES_CSV  = META_DIR / "genres.csv"
FEATURES_FMA = META_DIR / "features.csv"
OUT_FEATURES_TEST = OUTPUT_DIR / "features_house_test500.csv"
OUT_ERRORS_TEST   = OUTPUT_DIR / "features_house_test500_errors.csv"
OUT_FEATURES_FULL = OUTPUT_DIR / "features_house_full.csv"
OUT_ERRORS_FULL   = OUTPUT_DIR / "features_house_full_errors.csv"
FEATURES_V2_CSV   = OUTPUT_DIR / "features_V2.csv"

# Gates structure (bloquants)
assert TRACKS_CSV.exists(), f"tracks.csv manquant : {TRACKS_CSV}"
assert GENRES_CSV.exists(), f"genres.csv manquant : {GENRES_CSV}"
assert AUDIO_DIR.exists(),  f"AUDIO_DIR manquant : {AUDIO_DIR}"

print(f"BASE      : {BASE}")
print(f"AUDIO_DIR : {AUDIO_DIR}")
print(f"OUTPUT    : {OUTPUT_DIR}")
print("Structure dossiers : OK ✅")

In [ ]:
# C7
# 1.3. Paramètres figés + versions + seed

import sys, numpy as np

# --- A) Versions ---
print("Python  :", sys.version.split()[0])
print("pandas  :", pd.__version__)
print("numpy   :", np.__version__)
print("librosa :", librosa.__version__)

# --- B) Paramètres figés (hérités de l'EDA — ne pas modifier) ---
SR         = 22050   # fréquence d'échantillonnage
MONO       = True    # conversion mono
DURATION   = 30.0    # fenêtre d'analyse en secondes
SEED       = 42      # reproductibilité
N_MFCC     = 20      # nombre de coefficients MFCC
N_TEST     = 500     # taille échantillon validation pipeline
BATCH_SIZE = 200     # checkpoint toutes les N pistes

#  --- C) Comptage features (7 stats sur tous les groupes multi-dim, tempo scalaire)
# mfcc 20×7=140
# chroma_stft 12×7=84
# centroid/bandwidth/rolloff 3×(1×7)=21
# contrast 7×7=49
# rmse 1×7=7
# tonnetz 6×7=42
# zcr 1×7=7
# tempo 1
EXPECTED_N_FEATURES = 351

np.random.seed(SEED)
print(f"SR={SR} | MONO={MONO} | DURATION={DURATION}s | SEED={SEED} | N_MFCC={N_MFCC}")
print(f"N_TEST={N_TEST} | BATCH_SIZE={BATCH_SIZE} | EXPECTED_N_FEATURES={EXPECTED_N_FEATURES}")

# --- D) Stratégies figées (transmises Phase 3)
SUBGENRES_STRATEGY = {"default": "top-k", "k": 3, "multi_label_if_time": True, "freq_threshold": 50}
SPLIT_STRATEGY     = {"preferred": "group_split_by_artist", "test_size": 0.2, "random_state": 42,
                      "fallback": "stratified_split_genre_top_and_document_bias"}
EVAL_STRATEGY      = {"primary_metric": "macro_f1", "secondary_metric": "accuracy",
                      "report_by_class": True, "confusion_matrix": True}

print()
print("Stratégies figées (EDA C27) :")
print("  SUBGENRES_STRATEGY :", SUBGENRES_STRATEGY)
print("  SPLIT_STRATEGY     :", SPLIT_STRATEGY)
print("  EVAL_STRATEGY      :", EVAL_STRATEGY)

In [ ]:
# C8
# 2. Chargement tracks.csv + construction labels_df complet
# labels_df est la source de vérité pour tous les labels Phase 3
# Construit AVANT l'extraction pour garantir la complétude du CSV final

# --- A) Chargement tracks.csv + filtrage Small ---
tracks       = pd.read_csv(TRACKS_CSV, index_col=0, header=[0, 1])
small_tracks = tracks[tracks[("set", "subset")] == "small"]
small_ids    = small_tracks.index.astype(int)

print("tracks shape       :", tracks.shape)
print("small_tracks shape :", small_tracks.shape)
assert small_tracks.shape[0] == 8000, f"Attendu 8000 pistes Small, obtenu {small_tracks.shape[0]}"

# --- B) Chargement genres.csv → dictionnaire id_to_title ---
genres_ref = pd.read_csv(GENRES_CSV, index_col=0)
id_to_title = genres_ref["title"].to_dict()
print("genres_ref shape   :", genres_ref.shape)

# --- C) Décodage sous-genres (identique EDA C16) ---
def decode_genres(x):
    """Convertit une string '[21, 17]' en liste de titres de sous-genres."""
    try:
        ids = ast.literal_eval(str(x))
        return [id_to_title.get(i, str(i)) for i in ids]
    except Exception:
        return []

genres_decoded = small_tracks[("track", "genres")].apply(decode_genres)

# --- D) Calcul mismatch (identique EDA C18) ---
# mismatch = genre_top absent de genres_decoded
# Indicateur d'ambiguïté/hiérarchie — pas preuve de mauvais étiquetage
genre_top_series = small_tracks[("track", "genre_top")].astype(str)
mismatch_series  = [
    gt not in lst
    for gt, lst in zip(genre_top_series, genres_decoded)
]

# --- E) Construction labels_df (index = track_id) ---
labels_df = pd.DataFrame({
    "genre_top"      : small_tracks[("track", "genre_top")].values,
    "genres"         : small_tracks[("track", "genres")].values,
    "genres_decoded" : genres_decoded.values,
    "n_subgenres"    : genres_decoded.apply(len).values,
    "mismatch"       : mismatch_series,
    "artist_name"    : small_tracks[("artist", "name")].values,
    "track_title"    : small_tracks[("track", "title")].values,
    "year"           : pd.to_datetime(
                           small_tracks[("track", "date_created")], errors="coerce"
                       ).dt.year.values,
    "duration"       : pd.to_numeric(
                           small_tracks[("track", "duration")], errors="coerce"
                       ).values,
    "bit_rate"       : pd.to_numeric(
                           small_tracks[("track", "bit_rate")], errors="coerce"
                       ).values,
}, index=small_ids)
labels_df.index.name = "track_id"

# --- F) Validation labels_df ---
print("\nlabels_df shape :", labels_df.shape)
print("Colonnes        :", labels_df.columns.tolist())
print("NA par colonne  :")
print(labels_df.isnull().sum())
print("\nmismatch global (%) :", round(labels_df["mismatch"].mean() * 100, 1))
print("n_subgenres median  :", labels_df["n_subgenres"].median())
print("\nDistribution genre_top :")
print(labels_df["genre_top"].value_counts())

assert labels_df.shape[0] == 8000,       "labels_df : attendu 8000 lignes"
assert labels_df["genre_top"].isna().sum() == 0, "genre_top : valeurs manquantes"
assert labels_df["artist_name"].isna().sum() == 0, "artist_name : valeurs manquantes"
print("\nValidation labels_df : OK ✅")

# --- G) Sous-genres éligibles (freq_threshold=50, identique EDA C27) ---
# Calcul effectué ici pour que Phase 3 puisse l'utiliser directement
all_sub      = [g for sublist in labels_df["genres_decoded"] 
                if isinstance(sublist, list) for g in sublist]
sub_counts   = pd.Series(all_sub).value_counts()
subgenres_eligible = sub_counts[sub_counts >= SUBGENRES_STRATEGY["freq_threshold"]]
print(
    f"\nSous-genres >= {SUBGENRES_STRATEGY['freq_threshold']} occurrences : "
    f"{len(subgenres_eligible)}"
)
print(subgenres_eligible.head(30))

In [ ]:
# C9
# 3. Validation environnement audio (gate pré-extraction)
# Identique EDA C12 — gate bloquante : audio_decode_ok doit être True

# --- A) Construction mp3_map (os.walk — plus fiable que rglob sur Windows) ---
mp3_map: Dict[int, str] = {}
for root, _, files in os.walk(AUDIO_DIR):
    for f in files:
        if f.lower().endswith(".mp3"):
            try:
                tid = int(Path(f).stem)
            except ValueError:
                continue
            mp3_map[tid] = str(Path(root) / f)

print(f"mp3_map size : {len(mp3_map)}")
assert len(mp3_map) == 8000, f"Attendu 8000 MP3, obtenu {len(mp3_map)}"

# --- B) Gate décodage audio (échantillon déterministe seed=42) ---
def audio_decode_gate(mp3_map: Dict[int, str], n_test: int = 5,
                      seed: int = 42, sr: int = 22050,
                      duration_s: float = 5.0) -> dict:
    ids_sorted = np.array(sorted(mp3_map.keys()), dtype=int)
    rng        = np.random.default_rng(seed)
    sample_ids = rng.choice(ids_sorted, size=min(n_test, len(ids_sorted)), replace=False)

    errors     = 0
    first_error = None
    for tid in sample_ids:
        try:
            y, _ = librosa.load(mp3_map[int(tid)], mono=True, sr=sr, duration=duration_s)
            if y is None or len(y) == 0:
                errors += 1
                if first_error is None:
                    first_error = f"Empty audio : track_id={tid}"
        except Exception as e:
            errors += 1
            if first_error is None:
                first_error = f"track_id={tid} -> {repr(e)}"

    return {
        "audio_decode_ok" : (errors == 0),
        "n_test"          : n_test,
        "errors"          : errors,
        "first_error"     : first_error,
        "sr"              : sr,
        "duration_s"      : duration_s,
        "seed"            : seed,
    }

# --- C) Exécution gate ---
res_gate = audio_decode_gate(mp3_map, n_test=5, seed=SEED, sr=SR, duration_s=5.0)
print(f"Erreurs décodage  : {res_gate['errors']} / {res_gate['n_test']}")
print(f"audio_decode_ok   : {res_gate['audio_decode_ok']}")

if not res_gate["audio_decode_ok"]:
    print("\nFirst error :", res_gate["first_error"])
    print("Actions : pip install librosa audioread soundfile + installer ffmpeg")

# Gate bloquante
assert res_gate["audio_decode_ok"], "Décodage audio KO — impossible de continuer"
print("\nEnvironnement audio : OK ✅ — extraction autorisée")

In [ ]:
# C10
# 4. Fonctions extraction
# extract_features   : extrait 351 features depuis un fichier MP3
# compute_stats      : 7 stats (mean/std/min/max/median/skew/kurtosis) par coefficient
# run_extraction     : runner générique — resume via CSV, batch append, log erreurs
# deterministic_sample : échantillon déterministe (tri préalable — stabilité cross-runs)
# load_done_ids      : idempotence basée sur le CSV écrit (pas de JSON corruptible)
# append_csv         : écriture append-safe par batch

import warnings
warnings.filterwarnings("ignore", message="Trying to estimate tuning from empty frequency set")

# --- A) compute_stats : 7 stats identiques à FMA ---
def compute_stats(arr: np.ndarray) -> dict:
    """Calcule 7 stats sur chaque ligne d'un tableau 1D ou 2D.
    Retourne dict[num_str] = dict[stat_name] = float
    """
    if arr.ndim == 1:
        arr = arr.reshape(1, -1)
    result = {}
    for i, row in enumerate(arr):
        num = f"{i+1:02d}"
        result[num] = {
            "mean"     : float(np.mean(row)),
            "std"      : float(np.std(row)),
            "min"      : float(np.min(row)),
            "max"      : float(np.max(row)),
            "median"   : float(np.median(row)),
            "skew"     : float(scipy_stats.skew(row)),
            "kurtosis" : float(scipy_stats.kurtosis(row)),
        }
    return result

# --- B) extract_features : 351 features par piste ---
# Groupes : mfcc(20), chroma_stft(12), spectral_centroid(1), spectral_bandwidth(1),
#           spectral_rolloff(1), spectral_contrast(7), rmse(1), tonnetz(6), zcr(1), tempo(1)
# Nommage : {groupe}_{num:02d}_{stat} ex: mfcc_01_mean
# tonnetz : try/except → NaN si signal insuffisant (pas de crash pipeline)
# tempo   : beat_track (librosa 0.11 compatible)
def extract_features(path: str, sr: int = SR, mono: bool = MONO,
                     duration: float = DURATION, n_mfcc: int = N_MFCC) -> dict:
    """Extrait 351 features audio depuis un fichier MP3."""
    y, sr_ = librosa.load(path, mono=mono, sr=sr, duration=duration)
    features = {}

    # -- mfcc (20 coefficients × 7 stats = 140) --
    mfcc = librosa.feature.mfcc(y=y, sr=sr_, n_mfcc=n_mfcc)
    for num, s in compute_stats(mfcc).items():
        for stat, val in s.items():
            features[f"mfcc_{num}_{stat}"] = val

    # -- chroma_stft (12 coefficients × 7 stats = 84) --
    chroma = librosa.feature.chroma_stft(y=y, sr=sr_)
    for num, s in compute_stats(chroma).items():
        for stat, val in s.items():
            features[f"chroma_stft_{num}_{stat}"] = val

    # -- spectral_centroid (1 × 7 = 7) --
    sc = librosa.feature.spectral_centroid(y=y, sr=sr_)
    for num, s in compute_stats(sc).items():
        for stat, val in s.items():
            features[f"spectral_centroid_{num}_{stat}"] = val

    # -- spectral_bandwidth (1 × 7 = 7) --
    sb = librosa.feature.spectral_bandwidth(y=y, sr=sr_)
    for num, s in compute_stats(sb).items():
        for stat, val in s.items():
            features[f"spectral_bandwidth_{num}_{stat}"] = val

    # -- spectral_rolloff (1 × 7 = 7) --
    sr_feat = librosa.feature.spectral_rolloff(y=y, sr=sr_)
    for num, s in compute_stats(sr_feat).items():
        for stat, val in s.items():
            features[f"spectral_rolloff_{num}_{stat}"] = val

    # -- spectral_contrast (7 bandes × 7 stats = 49) --
    contrast = librosa.feature.spectral_contrast(y=y, sr=sr_)
    for num, s in compute_stats(contrast).items():
        for stat, val in s.items():
            features[f"spectral_contrast_{num}_{stat}"] = val

    # -- rmse / rms (1 × 7 = 7) --
    rms = librosa.feature.rms(y=y)
    for num, s in compute_stats(rms).items():
        for stat, val in s.items():
            features[f"rmse_{num}_{stat}"] = val

    # -- tonnetz (6 dim × 7 stats = 42) --
    # Nécessite signal harmonique — peut échouer sur certaines pistes
    # → NaN plutôt que crash du pipeline
    try:
        y_harm  = librosa.effects.harmonic(y)
        tonnetz = librosa.feature.tonnetz(y=y_harm, sr=sr_)
        for num, s in compute_stats(tonnetz).items():
            for stat, val in s.items():
                features[f"tonnetz_{num}_{stat}"] = val
    except Exception:
        for i in range(1, 7):
            for stat in ["mean", "std", "min", "max", "median", "skew", "kurtosis"]:
                features[f"tonnetz_{i:02d}_{stat}"] = float("nan")

    # -- zcr (1 × 7 = 7) --
    zcr = librosa.feature.zero_crossing_rate(y)
    for num, s in compute_stats(zcr).items():
        for stat, val in s.items():
            features[f"zcr_{num}_{stat}"] = val

    # -- tempo (1) — via beat_track, compatible librosa 0.11 --
    tempo_raw, _ = librosa.beat.beat_track(y=y, sr=sr_)
    if isinstance(tempo_raw, np.ndarray):
        tempo = float(tempo_raw.squeeze())
    else:
        tempo = float(tempo_raw)
    features["tempo"] = tempo

    return features

# --- C) Helpers idempotence + batch ---

def deterministic_sample(ids: List[int], n: int, seed: int) -> List[int]:
    """Échantillon déterministe stable : tri préalable obligatoire."""
    ids_sorted = np.array(sorted(ids), dtype=int)
    rng        = np.random.default_rng(seed)
    if n > len(ids_sorted):
        raise ValueError(f"n={n} > nb ids disponibles={len(ids_sorted)}")
    sample = rng.choice(ids_sorted, size=n, replace=False)
    return sample.astype(int).tolist()

def load_done_ids(features_csv: Path) -> set:
    """Idempotence basée sur le CSV écrit — pas de JSON corruptible."""
    if not features_csv.exists():
        return set()
    prev = pd.read_csv(features_csv, usecols=["track_id"])
    return set(prev["track_id"].astype(int).tolist())

def append_csv(df: pd.DataFrame, out_csv: Path) -> None:
    """Écriture append-safe : header uniquement si fichier n'existe pas encore."""
    if out_csv.exists():
        df.to_csv(out_csv, mode="a", header=False, index=False)
    else:
        df.to_csv(out_csv, index=False)

def write_errors(errors: List[dict], out_errors: Path) -> None:
    if len(errors) == 0:
        return
    pd.DataFrame(errors).to_csv(out_errors, index=False)

# --- D) extract_one : extraction + safety checks ---
def extract_one(track_id: int, path: str) -> dict:
    """Extrait les features d'une piste + vérifie le schéma."""
    feats = extract_features(path)
    feats["track_id"] = int(track_id)

    # Toutes les valeurs sauf track_id doivent être float
    non_float = {k: type(v) for k, v in feats.items()
                 if k != "track_id" and not isinstance(v, float)}
    assert len(non_float) == 0, f"Non-float détectés: {non_float}"
    return feats

# --- E) run_extraction : runner générique ---
def run_extraction(
    ids             : List[int],
    out_features    : Path,
    out_errors      : Path,
    batch_size      : int           = 200,
    resume          : bool          = True,
    expected_n_features : Optional[int] = None,
) -> Tuple[Path, Path]:
    """Extrait les features pour une liste de track_ids.
    Resume automatique depuis le CSV déjà écrit (idempotent).
    Batch append toutes les batch_size pistes.
    """
    out_features = Path(out_features)
    out_errors   = Path(out_errors)

    done = load_done_ids(out_features) if resume else set()
    print(f"[RESUME] already done: {len(done)}")

    rows   : List[dict] = []
    errors : List[dict] = []
    t0 = time.time()

    todo = [int(t) for t in ids if int(t) not in done]
    print(f"To process: {len(todo)}/{len(ids)}")

    for k, tid in enumerate(todo, start=1):
        p = mp3_map.get(int(tid))
        if p is None:
            errors.append({"track_id": int(tid), "error": "mp3_not_found", "traceback": ""})
            continue

        try:
            row = extract_one(int(tid), p)

            # Contrôle schéma si attendu
            if expected_n_features is not None:
                n_feats = len(row) - 1  # hors track_id
                assert n_feats == expected_n_features, \
                    f"track {tid}: {n_feats} features != attendu {expected_n_features}"

            rows.append(row)

        except Exception as e:
            errors.append({
                "track_id" : int(tid),
                "error"    : repr(e),
                "traceback": traceback.format_exc(limit=3)
            })

        # Checkpoint batch
        if len(rows) >= batch_size:
            append_csv(pd.DataFrame(rows), out_features)
            rows = []
            write_errors(errors, out_errors)

            elapsed   = time.time() - t0
            rate      = k / elapsed if elapsed > 0 else float("nan")
            remaining = (len(todo) - k) / rate if rate > 0 else float("nan")
            print(f"[checkpoint] {k}/{len(todo)} processed | "
                  f"elapsed {elapsed:.0f}s | rate {rate:.2f}/s | "
                  f"~{remaining:.0f}s left | errors {len(errors)}")

    # Flush final
    if len(rows) > 0:
        append_csv(pd.DataFrame(rows), out_features)
    write_errors(errors, out_errors)

    print(f"DONE. features : {out_features}")
    print(f"DONE. errors   : {out_errors}")
    return out_features, out_errors

print("Fonctions extraction chargées ✅")
print(f"EXPECTED_N_FEATURES = {EXPECTED_N_FEATURES}")

In [ ]:
# C11
# 5. Test unitaire — validation schéma sur 1 piste
# Bloque l'extraction si le schéma est incorrect
# Exécuter après C9 et avant C12

# --- A) Sélection piste de test (première du mp3_map trié) ---
_test_tid  = sorted(mp3_map.keys())[0]
_test_path = mp3_map[_test_tid]
print(f"=== TEST UNITAIRE (1 piste) ===")
print(f"track_id : {_test_tid}")
print(f"path     : {_test_path}")

# --- B) Extraction + validations ---
try:
    _feats = extract_features(_test_path)

    # Nb features
    print(f"\nNb features extraites : {len(_feats)}")
    print(f"Attendu               : {EXPECTED_N_FEATURES}")
    assert len(_feats) == EXPECTED_N_FEATURES, \
        f"Attendu {EXPECTED_N_FEATURES} features, obtenu {len(_feats)}"

    # Types : tout doit être float
    non_float = {k: type(v) for k, v in _feats.items() if not isinstance(v, float)}
    print(f"Non-float             : {non_float if non_float else 'aucun ✅'}")
    assert len(non_float) == 0, f"Non-float détectés : {non_float}"

    # NaN hors tonnetz
    nan_keys = [k for k, v in _feats.items()
                if "tonnetz" not in k and (v != v)]
    print(f"NaN hors tonnetz      : {nan_keys if nan_keys else 'aucun ✅'}")
    assert len(nan_keys) == 0, f"NaN inattendus hors tonnetz : {nan_keys}"

    # Spot-check valeurs
    print(f"\nmfcc_01_mean          : {_feats.get('mfcc_01_mean', 'MANQUANT'):.4f}")
    print(f"chroma_stft_01_mean   : {_feats.get('chroma_stft_01_mean', 'MANQUANT'):.4f}")
    print(f"tempo                 : {_feats.get('tempo', 'MANQUANT'):.2f} BPM")

    print("\nTest unitaire : OK ✅ — extraction autorisée")
    _unit_test_ok = True

except Exception as e:
    print(f"\nTest unitaire : ERREUR ❌ — {repr(e)}")
    _unit_test_ok = False

assert _unit_test_ok, "Test unitaire KO — corriger extract_features avant de continuer"

In [ ]:
# C12
# 6. Extraction test 500 pistes (échantillon déterministe seed=42)
# Valide le pipeline avant de lancer les 8000 pistes
# Idempotent : resume depuis OUT_FEATURES_TEST si déjà existant

# --- A) Échantillon déterministe (tri préalable — stabilité cross-runs) ---
test_ids = deterministic_sample(list(mp3_map.keys()), n=N_TEST, seed=SEED)
print(f"Échantillon test : {len(test_ids)} pistes (seed={SEED})")

# --- B) Extraction test 500 ---
out_f_test, out_e_test = run_extraction(
    ids                 = test_ids,
    out_features        = OUT_FEATURES_TEST,
    out_errors          = OUT_ERRORS_TEST,
    batch_size          = BATCH_SIZE,
    resume              = True,
    expected_n_features = EXPECTED_N_FEATURES,
)

# --- C) Validation test500 ---
df_test = pd.read_csv(OUT_FEATURES_TEST)
print(f"\n=== VALIDATION TEST500 ===")
print(f"Shape                  : {df_test.shape}")
print(f"track_id uniques       : {df_test['track_id'].nunique()}")
print(f"Colonnes (5 premières) : {df_test.columns.tolist()[:5]}")
print(f"Colonnes (5 dernières) : {df_test.columns.tolist()[-5:]}")
n_errors_test = pd.read_csv(OUT_ERRORS_TEST).shape[0] if OUT_ERRORS_TEST.exists() else 0
print(f"Erreurs                : {n_errors_test}")

assert df_test["track_id"].nunique() == N_TEST, \
    f"Attendu {N_TEST} track_ids uniques, obtenu {df_test['track_id'].nunique()}"
assert df_test.shape[1] == EXPECTED_N_FEATURES + 1, \
    f"Attendu {EXPECTED_N_FEATURES + 1} colonnes, obtenu {df_test.shape[1]}"
print("\nTest500 : OK ✅")

In [ ]:
# C13
# 7. Corrélation pipeline maison vs FMA (500 pistes)
# Valide que nos features sont cohérentes avec la référence FMA
# Documente l'écart MFCC (normalisation corpus-level FMA vs piste-level maison)

# --- A) Chargement features FMA (filtrage Small) ---
# Note : chargement uniquement pour la validation — pas utilisé dans le CSV final
print("Chargement features.csv FMA (peut prendre 20-30s)...")
assert FEATURES_FMA.exists(), "features.csv FMA manquant — section validation uniquement, pas bloquant"
t0 = time.time()
features_fma_full = pd.read_csv(FEATURES_FMA, index_col=0, header=[0, 1, 2])
features_fma      = features_fma_full.loc[features_fma_full.index.intersection(small_ids)]
print(f"Chargé en {time.time()-t0:.1f}s — shape Small : {features_fma.shape}")

# --- B) Pistes communes maison ∩ FMA ---
df_maison  = pd.read_csv(OUT_FEATURES_TEST, index_col="track_id")
common_ids = df_maison.index.intersection(features_fma.index)
print(f"Pistes communes maison ∩ FMA : {len(common_ids)}")
assert len(common_ids) >= 400, f"Trop peu de pistes communes : {len(common_ids)}"

# --- C) Corrélation de Pearson par groupe (mean uniquement) ---
GROUPS = {
    "mfcc"               : (20, "mfcc"),
    "chroma_stft"        : (12, "chroma_stft"),
    "spectral_centroid"  : (1,  "spectral_centroid"),
    "spectral_bandwidth" : (1,  "spectral_bandwidth"),
    "spectral_rolloff"   : (1,  "spectral_rolloff"),
    "spectral_contrast"  : (7,  "spectral_contrast"),
    "rmse"               : (1,  "rmse"),
    "tonnetz"            : (6,  "tonnetz"),
    "zcr"                : (1,  "zcr"),
}

results_corr = {}
for group_name, (n_coeff, fma_group) in GROUPS.items():
    corrs = []
    for i in range(1, n_coeff + 1):
        num        = f"{i:02d}"
        col_maison = f"{group_name}_{num}_mean"
        if col_maison not in df_maison.columns:
            continue
        try:
            fma_col = features_fma.loc[common_ids, (fma_group, "mean", num)]
        except KeyError:
            continue
        maison_col = df_maison.loc[common_ids, col_maison]
        mask = maison_col.notna() & fma_col.notna()
        if mask.sum() < 10:
            continue
        r = float(np.corrcoef(maison_col[mask].values, fma_col[mask].values)[0, 1])
        corrs.append(r)
    if corrs:
        results_corr[group_name] = {
            "n_coeffs" : len(corrs),
            "median_r" : round(float(np.median(corrs)), 4),
            "min_r"    : round(float(np.min(corrs)), 4),
            "max_r"    : round(float(np.max(corrs)), 4),
        }

# --- D) Affichage résultats ---
print(f"\n=== CORRÉLATIONS MAISON vs FMA (mean, {len(common_ids)} pistes) ===")
print(f"{'Groupe':<25} {'n_coeffs':>8} {'median_r':>10} {'min_r':>8} {'max_r':>8}")
print("-" * 65)
for g, v in results_corr.items():
    print(f"{g:<25} {v['n_coeffs']:>8} {v['median_r']:>10.4f} {v['min_r']:>8.4f} {v['max_r']:>8.4f}")

all_medians   = [v["median_r"] for v in results_corr.values()]
global_median = float(np.median(all_medians))
print(f"\nCorrélation médiane globale : {global_median:.4f}")

# --- E) Conclusion documentée ---
# L'écart MFCC (corr ~0.49) est structurel, pas un bug :
# FMA a calculé ses features sur 106 574 pistes (normalisation corpus-level)
# Notre extraction est piste-level → décalage absolu non reproductible
# Après StandardScaler en Phase 3, les valeurs absolues n'ont aucun impact
print("\n=== CONCLUSION ===")
print("Écart MFCC (~0.49) : normalisation corpus-level FMA vs piste-level maison")
print("Non reproductible sans le code source FMA + 106 574 pistes")
print("Non bloquant : StandardScaler Phase 3 neutralise les décalages absolus")
print("Pipeline maison : VALIDÉ ✅ — extraction complète autorisée")

In [ ]:
# C14
# 8. Extraction complète — 8 000 pistes  + gate test500
# Idempotent : resume depuis OUT_FEATURES_FULL si interruption
# Perte max en cas de crash : BATCH_SIZE pistes (200)
# Avant de lancer : brancher sur secteur, désactiver mise en veille Windows

already_done = load_done_ids(OUT_FEATURES_FULL)
n_expected   = len(mp3_map)

# --- A) Vérification pré-lancement ---
print("=== PRÉ-LANCEMENT FULL 8000 ===")
print(f"AUDIO_DIR    : {AUDIO_DIR}")
print(f"OUTPUT_DIR   : {OUTPUT_DIR}")
print(f"mp3_map size : {len(mp3_map)}")
print(f"BATCH_SIZE   : {BATCH_SIZE}")
print(f"SEED         : {SEED}")

# Estimation temps basée sur le test500 (~0.37 pistes/s)
RATE_OBS  = 0.37   # pistes/s observé sur test500
eta_hours = (len(mp3_map) / RATE_OBS) / 3600
print(f"\nEstimation durée : ~{eta_hours:.1f}h (basée sur test500 à {RATE_OBS} pistes/s)")
print(f"Checkpoint toutes les {BATCH_SIZE} pistes → résistant aux interruptions")

# Resume : pistes déjà extraites
print(f"\nPistes déjà extraites (resume) : {len(already_done)}")
print(f"Reste à traiter                : {len(mp3_map) - len(already_done)}")

# Gate bloquante : test500 doit être validé
assert OUT_FEATURES_TEST.exists(), "Test500 manquant — relancer C11 d'abord"
df_gate = pd.read_csv(OUT_FEATURES_TEST)
assert df_gate["track_id"].nunique() == N_TEST, "Test500 incomplet"
print("\nGate test500 : OK ✅ — lancement full autorisé")

# --- B) Gate idempotente — skip si extraction déjà complète ou fichiers 'features_house_full.csv' et 'features_V2.csv' déjà existants ---
if len(already_done) >= n_expected - 10:
    print(f"Extraction déjà complète ({len(already_done)}/{n_expected} pistes) — skip ✅")
    print(f"CSV disponible : {OUT_FEATURES_FULL}")
    print("Passer directement à C15.")
else:
    print(f"Pistes manquantes : {n_expected - len(already_done)} — lancement...")

    target_ids = sorted(mp3_map.keys())
    out_f_full, out_e_full = run_extraction(
        ids                 = target_ids,
        out_features        = OUT_FEATURES_FULL,
        out_errors          = OUT_ERRORS_FULL,
        batch_size          = BATCH_SIZE,
        resume              = True,
        expected_n_features = EXPECTED_N_FEATURES,
    )

# --- C) Validation post-extraction ---
df_full       = pd.read_csv(OUT_FEATURES_FULL)
n_errors_full = pd.read_csv(OUT_ERRORS_FULL).shape[0] if OUT_ERRORS_FULL.exists() else 0
n_missing     = len(mp3_map) - df_full["track_id"].nunique()

print(f"\n=== VALIDATION FULL ===")
print(f"Shape             : {df_full.shape}")
print(f"track_id uniques  : {df_full['track_id'].nunique()} / {n_expected}")
print(f"Pistes manquantes : {n_missing}")
print(f"Erreurs loggées   : {n_errors_full}")

if n_missing > 0:
    print(f"\n⚠️  {n_missing} pistes manquantes → lancer C15 (retry ciblé)")
else:
    print("\nExtraction full : OK ✅")

In [ ]:
# C15
# 9. Retry ciblé — pistes manquantes + erreurs
# Idempotent : si toutes les pistes sont extraites, ne fait rien
# À lancer après C13 si n_missing > 0

# --- A) Identification pistes à retenter ---
df_full   = pd.read_csv(OUT_FEATURES_FULL)
done_ids  = set(df_full["track_id"].astype(int).tolist())
expected  = set(mp3_map.keys())

missing_ids = sorted(expected - done_ids)
print(f"Pistes manquantes (vs mp3_map) : {len(missing_ids)}")
if missing_ids:
    print(f"IDs manquants : {missing_ids}")

error_ids = []
if OUT_ERRORS_FULL.exists():
    df_err    = pd.read_csv(OUT_ERRORS_FULL)
    print(f"\nErreurs loggées : {len(df_err)}")
    print(df_err[["track_id", "error"]].to_string())
    error_ids = sorted(df_err["track_id"].dropna().astype(int).unique().tolist())

retry_ids = sorted(set(missing_ids) | set(error_ids))
print(f"\nRetry total : {len(retry_ids)} pistes")

# --- B) Retry si nécessaire ---
if len(retry_ids) > 0:
    out_f_full, out_e_full = run_extraction(
        ids                 = retry_ids,
        out_features        = OUT_FEATURES_FULL,
        out_errors          = OUT_ERRORS_FULL,
        batch_size          = BATCH_SIZE,
        resume              = True,
        expected_n_features = EXPECTED_N_FEATURES,
    )
else:
    print("Aucune piste à retenter ✅")

# --- C) Validation post-retry ---
df_full  = pd.read_csv(OUT_FEATURES_FULL)
n_final  = df_full["track_id"].nunique()
n_errors = pd.read_csv(OUT_ERRORS_FULL).shape[0] if OUT_ERRORS_FULL.exists() else 0

print(f"\n=== VALIDATION APRÈS RETRY ===")
print(f"track_id uniques : {n_final} / {len(mp3_map)}")
print(f"Erreurs résiduelles : {n_errors}")

# Les erreurs résiduelles sont des MP3 corrompus — documentés, non bloquants
if n_errors > 0:
    print(f"⚠️  {n_errors} MP3 corrompus — documentés dans {OUT_ERRORS_FULL.name}")
    print("   Ces pistes seront absentes du CSV final — non bloquant pour la modélisation")
print(f"\nExtraction : {n_final} pistes disponibles ✅")

In [ ]:
# C16
# 10. Construction du CSV final — jointure features + labels
# Principe : labels_df (construit en C7) est la source de vérité
# Jointure left sur track_id → features_V2 contient features + tous les labels Phase 3

# --- A) Rechargement df_full depuis disque (idempotence — indépendant de la RAM) ---
df_full = pd.read_csv(OUT_FEATURES_FULL)
df_full["track_id"] = df_full["track_id"].astype(int)
print(f"df_full shape    : {df_full.shape}")
print(f"track_id uniques : {df_full['track_id'].nunique()}")

# Vérification doublons track_id (ne doit pas y en avoir)
n_dup = df_full["track_id"].duplicated().sum()
print(f"Doublons track_id : {n_dup}")
assert n_dup == 0, f"{n_dup} doublons track_id détectés — vérifier run_extraction"

# --- B) Préparation labels_df pour jointure ---
# genres_decoded est une liste Python — convertie en string repr pour le CSV
# IMPORTANT Phase 3 : utiliser ast.literal_eval(x) pour récupérer la liste
# Exemple : ast.literal_eval(df["genres_decoded"].iloc[0]) → ["Punk", "Indie-Rock"]
labels_for_join = labels_df.copy()
labels_for_join["genres_decoded"] = labels_for_join["genres_decoded"].apply(
    lambda x: str(x) if isinstance(x, list) else "[]"
)
labels_for_join = labels_for_join.reset_index()  # track_id devient colonne

# --- C) Jointure features + labels ---
df_v2 = df_full.merge(
    labels_for_join,
    on       = "track_id",
    how      = "left",
    validate = "one_to_one",
)

print(f"\ndf_v2 shape après jointure : {df_v2.shape}")

# --- D) Réorganisation colonnes ---
# Ordre : track_id | labels | features audio
label_cols = ["track_id", "genre_top", "genres", "genres_decoded", "n_subgenres",
              "mismatch", "artist_name", "track_title", "year", "duration", "bit_rate"]
feature_cols = [c for c in df_full.columns if c != "track_id"]

# Vérification colonnes attendues présentes
missing_label_cols = [c for c in label_cols if c not in df_v2.columns]
assert len(missing_label_cols) == 0, f"Colonnes labels manquantes : {missing_label_cols}"

df_v2 = df_v2[label_cols + feature_cols]

# --- E) Validation finale ---
print(f"\n=== VALIDATION df_v2 ===")
print(f"Shape                  : {df_v2.shape}")
print(f"Colonnes labels        : {label_cols}")
print(f"Nb features audio      : {len(feature_cols)}")
print(f"NA genre_top           : {df_v2['genre_top'].isna().sum()}")
print(f"NA artist_name         : {df_v2['artist_name'].isna().sum()}")
print(f"NA mismatch            : {df_v2['mismatch'].isna().sum()}")
print(f"\nDistribution genre_top :")
print(df_v2["genre_top"].value_counts())
print(f"\nmismatch global (%)    : {round(df_v2['mismatch'].mean() * 100, 1)}")

assert df_v2["genre_top"].isna().sum() == 0,   "genre_top : valeurs manquantes"
assert df_v2["artist_name"].isna().sum() == 0, "artist_name : valeurs manquantes"
assert len(feature_cols) == EXPECTED_N_FEATURES, \
    f"Attendu {EXPECTED_N_FEATURES} features audio, obtenu {len(feature_cols)}"

print("\nConstruction df_v2 : OK ✅")

# --- F) Export idempotent du livrable features_V2.csv ---
df_v2.to_csv(FEATURES_V2_CSV, index=False)
file_size_mb = FEATURES_V2_CSV.stat().st_size / (1024 * 1024)
print(f"\nExport V2 : {FEATURES_V2_CSV}")
print(f"Taille    : {file_size_mb:.1f} Mo")

In [ ]:
# C17
# 11. Export features_V2.csv + check final (LIVRABLE)
# Idempotent : relancer réécrit le même CSV si pipeline inchangé

# --- Load + nettoyage colonnes parasites ---
assert FEATURES_V2_CSV.exists(), f"CSV introuvable : {FEATURES_V2_CSV} (exécuter C16 avant C17)"
df_v2 = pd.read_csv(FEATURES_V2_CSV)
df_v2 = df_v2.loc[:, ~df_v2.columns.astype(str).str.startswith("Unnamed:")]

label_cols = [
    "track_id", "genre_top", "genres", "genres_decoded", "n_subgenres",
    "mismatch", "artist_name", "track_title", "year", "duration", "bit_rate"
]
missing = [c for c in label_cols if c not in df_v2.columns]
assert not missing, f"Colonnes labels manquantes: {missing}"

# Normalisations Phase 3
df_v2["genres_decoded"] = df_v2["genres_decoded"].fillna("[]").astype(str)
df_v2["mismatch"] = pd.to_numeric(df_v2["mismatch"], errors="coerce").fillna(0).astype(int)

# Features count
feature_cols = [c for c in df_v2.columns if c not in label_cols]
assert len(feature_cols) == EXPECTED_N_FEATURES, (
    f"n_features={len(feature_cols)} != EXPECTED_N_FEATURES={EXPECTED_N_FEATURES}"
)

# --- Gate NA : diagnostic complet + blocage seulement sur critiques ---
na_all = df_v2[label_cols].isna().sum().sort_values(ascending=False)
print("\nNA par label (diagnostic):")
print(na_all.to_string())

crit_gate = ["track_id", "genre_top", "artist_name", "genres_decoded", "n_subgenres", "mismatch"]
na_crit = df_v2[crit_gate].isna().sum()
assert na_crit.sum() == 0, f"NA sur labels critiques: {na_crit[na_crit>0].to_dict()}"

# --- Export idempotent ---
df_v2.to_csv(FEATURES_V2_CSV, index=False)
size_mb = FEATURES_V2_CSV.stat().st_size / (1024 * 1024)
print(f"\nExport OK : {FEATURES_V2_CSV} | {size_mb:.1f} Mo")

# --- Rapport court (rapport-ready) ---
try:
    denom = len(mp3_map)
except Exception:
    denom = "NA"

print("\n=== CHECK FINAL features_V2.csv ===")
print(f"Pistes extraites : {df_v2.shape[0]} / {denom}")
print(f"Features audio   : {len(feature_cols)}")
print(f"Labels           : {len(label_cols)}")
print(f"Total colonnes   : {df_v2.shape[1]}")

tonnetz_cols = [c for c in feature_cols if "tonnetz" in c]
other_cols = [c for c in feature_cols if "tonnetz" not in c]
print(f"NaN tonnetz      : {int(df_v2[tonnetz_cols].isnull().sum().sum()) if tonnetz_cols else 0}")
print(f"NaN hors tonnetz : {int(df_v2[other_cols].isnull().sum().sum()) if other_cols else 0}")

print("\nDistribution genre_top :")
print(df_v2["genre_top"].value_counts().to_string())

print("\nMismatch % par genre :")
print((df_v2.groupby("genre_top")["mismatch"].mean()*100).round(1).sort_values().to_string())

# Log erreurs extraction (si dispo)
if "OUT_ERRORS_FULL" in globals() and OUT_ERRORS_FULL.exists():
    df_err = pd.read_csv(OUT_ERRORS_FULL)
    print(f"\nMP3 en erreur : {len(df_err)}")
    cols = [c for c in ["track_id", "error"] if c in df_err.columns]
    if cols:
        print(df_err[cols].head(30).to_string(index=False))
        if len(df_err) > 30:
            print("... (tronqué)")
else:
    print("\nMP3 en erreur : NA/0 (log absent ou variable non définie)")

# Re-load check
df_check = pd.read_csv(FEATURES_V2_CSV)
df_check = df_check.loc[:, ~df_check.columns.astype(str).str.startswith("Unnamed:")]
assert df_check.shape == df_v2.shape, f"Shape mismatch disque/mémoire: {df_check.shape} vs {df_v2.shape}"
assert list(df_check.columns) == list(df_v2.columns), "Colonnes mismatch disque/mémoire"

print("\nRechargement OK ✅ | features_V2.csv = LIVRABLE FINAL ✅")

---

## Conclusion — Phase 2 terminée

---

### Ce que ce notebook a produit

- **`features_V2.csv`** : dataset final prêt pour la modélisation
  - ~7994 pistes (6 MP3 corrompus non extractibles — documentés)
  - 351 features audio (10 extracteurs : 9 groupes × 7 stats + tempo scalaire)
  - 11 colonnes labels : track_id, genre_top, genres, genres_decoded, n_subgenres, mismatch, artist_name, track_title, year, duration, bit_rate

- **`features_house_full_errors.csv`** : log des pistes en erreur (MP3 corrompus)

---

### Décisions transmises à la Phase 3

| Décision | Valeur |
|----------|--------|
| Variable cible tâche A | `genre_top` (8 classes) |
| Variable cible tâche B | `genres_decoded` → top-k (k=3) ou multi-label (seuil ≥ 50) |
| Analyse transverse | `mismatch` — indicateur d'ambiguïté/hiérarchie |
| Split | Group split par `artist_name` (préféré) — seed=42 |
| Normalisation | StandardScaler avant tout modèle (neutralise décalages absolus) |
| Métrique primaire | macro F1 |
| Métrique secondaire | accuracy + confusion matrix |

---

### Note sur l'écart MFCC vs FMA

La corrélation MFCC maison vs FMA est ~0.49 — écart structurel expliqué :  
FMA a calculé ses features sur le corpus complet (106 574 pistes) avec une normalisation corpus-level.  
Notre extraction est piste-level. Non reproductible sans leur code source.  
**Non bloquant** : les modèles ML apprennent des patterns relatifs entre genres. StandardScaler neutralise les décalages absolus.

---

### Mode d’emploi Phase 3 — points techniques (à ne pas oublier)

**1. Parser 'genres_decoded' (stocké en string dans le CSV)**
Dans 'features_V2.csv', 'genres_decoded' est exporté comme une représentation texte de liste (ex. '"['Punk', 'Indie-Rock']"').
En Phase 3, il faut le reconvertir en liste Python :

**import ast**
**df["genres_decoded"] = df["genres_decoded"].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else [])**

**2. Split “artist-aware” (préféré) pour éviter la fuite train/test**
Utiliser 'artist_name' comme variable de groupe (GroupShuffleSplit ou GroupKFold).
Si ce split n’est pas faisable (ex. artistes manquants), documenter la limite et basculer en split stratifié 'genre_top'.

**3. Métriques standardisées (comparables entre modèles)**
- Toujours reporter **macro-F1** (principal) + **accuracy** (secondaire)
- **classification_report** par genre + **matrice de confusion**
- Conserver les **mêmes indices** (train/val/test) pour comparer LR/SVM vs modèle fort vs DL.

**4. Tâche B (sous-genres) : deux sorties possibles, une seule à retenir selon timing**
- **Top-k** : produire les k sous-genres les plus probables (k=3)
- **Multi-label** : conserver seulement les sous-genres avec fréquence ≥ 50, puis One-vs-Rest
Dans les deux cas, relier les erreurs au 'mismatch' (analyse transverse).

**5. Option DL (si retenue) : protocole minimal mais propre**
- Entrée : **log-mel spectrogram** calculé avec les mêmes paramètres audio (sr, mono, window)
- Split : **identique** au ML tabulaire (mêmes groupes artistes)
- Reporting : courbes train/val + macro-F1 + confusion matrix
(Si vous utilisez des segments, faire une agrégation track-level explicite et documentée.)

---
### Next step — Phase 3 : Modélisation

Lancer `NOTEBOOK_MODELISATION.ipynb` :
- Charger 'features_V2.csv' (features + labels)
- Reparser 'genres_decoded' (liste stockée en string) avec 'ast.literal_eval'
- Split **artist-aware** via 'artist_name' (seed=42) si possible ; sinon split stratifié et **limite documentée**
- Prétraitement : 'StandardScaler' sur les **351 features** (indispensable pour LR/SVM)
- Modèles (Tâche A — 'genre_top', 8 classes) :
- Baseline : **Logistic Regression** ou **Linear SVM**
- Modèle fort (1 seul, au choix, même protocole) : **XGBoost** (si retenu) / **HistGradientBoosting** (fallback sklearn) / **RandomForest** (fallback simple)
- Évaluation (comparaison stricte) :
- Métriques : **macro-F1** (primaire) + **accuracy** (secondaire)
- 'classification_report' par genre + **matrice de confusion**
- Même split / mêmes indices pour tous les modèles (comparabilité)
- Option (si temps) — Tâche B sous-genres + analyse mismatch :
- Sous-genres : **top-k (k=3)** ou **multi-label** (seuil ≥ 50)
- Relier erreurs/confusions au 'mismatch' (analyse transverse)
- Option (si temps) — DL comparatif :
- CNN sur log-mel (protocole d’éval identique : mêmes splits, mêmes métriques)

---

